# Dong Boundary Condition: Steady-State Convergence study computing Kovasznay Flow - Post processing

First test problem for validating the outflow B.C. by S. Dong. The exact solution for the velocity and pressure fields of the Kovasznay flow is given by:

>$$  \begin{align*} 
u &= 1 - \textrm{e}^{\lambda x} \cos{(2 \pi y)}, \\
v &= \frac{\lambda}{2 \pi} \textrm{e}^{\lambda x} \sin{(2 \pi y)}, \\
p &= \frac{1}{2} (1 - \textrm{e}^{2 \lambda x})
\end{align*}$$

where $\lambda = \frac{1}{2 \nu} - \sqrt{\frac{1}{4 \nu^2} + 4 \pi^2}$ with $\nu = \frac{1}{40}$.

In [1]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
using BoSSS.Solution.NSECommon;

In [3]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfAdditionalServiceCores,0
NumOfAdditionalServiceCoresMPISerial,0
NumOfServiceCoresPerMPIprocess,0
ServerName,DC3


In [4]:
BoSSSshell.WorkflowMgm.Init("KovasznayFlow_ConvStudy", myBatch);

Project name is set to 'KovasznayFlow_ConvStudy'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\KovasznayFlow_ConvStudy'.


In [55]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J327680_Picard	4/7/2025 9:09:31 AM	9a269a86...
#1: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p2_J786432_Picard	4/7/2025 9:07:31 AM	a5530912...
#2: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J81920_Picard	4/7/2025 9:09:01 AM	e78f6dc8...
#3: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p2_J196608_Picard	4/7/2025 9:06:42 AM	b838fbff...
#4: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J1310720_Picard*	4/7/2025 9:10:59 AM	9203bac7...
#5: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J20480_Picard	4/7/2025 9:08:46 AM	b8203c99...
#6: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p2_J49152_Picard	4/7/2025 9:06:21 AM	09ac1144...
#7: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J5120_Picard	4/7/2025 9:08:23 AM	e2a277ac...
#8: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J1280_Picard	4/7/2025 9:08:09 AM	bc742702...
#9: KovasznayFlow_ConvSt

In [56]:
int[] polyDeg = { 2, 3 };

In [57]:
var sess_polyDeg = sessions.Where(sess => sess.Name.Contains($"DongBC_KovasznayFlow_ConvStudy_p{polyDeg[1]}")
                                            && sess.SuccessfulTermination());
sess_polyDeg

#0: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J327680_Picard	4/7/2025 9:09:31 AM	9a269a86...
#1: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J81920_Picard	4/7/2025 9:09:01 AM	e78f6dc8...
#2: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J20480_Picard	4/7/2025 9:08:46 AM	b8203c99...
#3: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J5120_Picard	4/7/2025 9:08:23 AM	e2a277ac...
#4: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J1280_Picard	4/7/2025 9:08:09 AM	bc742702...
#5: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J320_Picard	4/7/2025 9:07:48 AM	2756177b...
#6: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J80_Picard	4/7/2025 9:07:34 AM	f42039f3...


In [58]:
var sess_polydeg_sort = sess_polyDeg.OrderBy(sess => sess.Timesteps.Last().Grid.NumberOfCells);
sess_polydeg_sort

#0: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J80_Picard	4/7/2025 9:07:34 AM	f42039f3...
#1: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J320_Picard	4/7/2025 9:07:48 AM	2756177b...
#2: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J1280_Picard	4/7/2025 9:08:09 AM	bc742702...
#3: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J5120_Picard	4/7/2025 9:08:23 AM	e2a277ac...
#4: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J20480_Picard	4/7/2025 9:08:46 AM	b8203c99...
#5: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J81920_Picard	4/7/2025 9:09:01 AM	e78f6dc8...
#6: KovasznayFlow_ConvStudy	DongBC_KovasznayFlow_ConvStudy_p3_J327680_Picard	4/7/2025 9:09:31 AM	9a269a86...


In [43]:
Formula KovasznayFlow_u = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velX = 1.0 - (Math.Exp(lambda * X[0]) * Math.Cos(2.0 * Math.PI * X[1]));" +
    "return velX; } "
);

In [13]:
Formula KovasznayFlow_v = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velY = (lambda/(2.0 * Math.PI)) * (Math.Exp(lambda * X[0]) * Math.Sin(2.0 * Math.PI * X[1]));" +
    "return velY; } "
);

In [14]:
Formula KovasznayFlow_p = new Formula(
    "Pres",
    false,
    "double Pres(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double Pres = 0.5 * (1.0 - Math.Exp(2.0 * lambda * X[0]));" +
    "return Pres; } "
);

In [59]:
int numGrids = sess_polydeg_sort.Count();
double[] numJ = new double[numGrids];
MultidimensionalArray L2norms = MultidimensionalArray.Create(3, numGrids);

for (int i = 0; i < numGrids; i++) {
    var ts = sess_polydeg_sort.Pick(i).Timesteps.Last();
    numJ[i] = ts.Grid.NumberOfCells;

    L2norms[0, i] = ts.GetField("VelocityX").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
        KovasznayFlow_u.EvaluateV(nodes, 0.0, results);
    });

    L2norms[1, i] = ts.GetField("VelocityY").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
        KovasznayFlow_v.EvaluateV(nodes, 0.0, results);
    });

    L2norms[2, i] = ts.GetField("Pressure").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
        KovasznayFlow_p.EvaluateV(nodes, 0.0, results);
    });
}

In [60]:
var gp = new Gnuplot();
gp.SetMultiplot(1,2);

Plot2Ddata plt = new Plot2Ddata(); 
var fmtX = new PlotFormat();    
fmtX.LineColor = LineColors.Red;
plt.AddDataGroup("VelocityX", numJ, L2norms.ExtractSubArrayShallow(0, -1).To1DArray(), fmtX);
var fmtY = new PlotFormat();    
fmtY.LineColor = LineColors.Blue;
plt.AddDataGroup("VelocityY", numJ, L2norms.ExtractSubArrayShallow(1, -1).To1DArray(), fmtY);
plt.LogX = true;
plt.LogY = true;
gp.SetSubPlot(0, 0, "L2 norms - velocities");
plt.ToGnuplot(gp);

plt = new Plot2Ddata(); 
var fmtP = new PlotFormat();    
fmtP.LineColor = LineColors.Green;
plt.AddDataGroup("Pressure", numJ, L2norms.ExtractSubArrayShallow(2, -1).To1DArray(), fmtP);
plt.LogX = true;
plt.LogY = true;
gp.SetSubPlot(0, 1, "L2 norms - pressure");
plt.ToGnuplot(gp);

//gp.PlotNow()
gp.PlotSVG(xRes:1500,yRes:500)

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 10 3 
 
 
 
 
 10 4 
 
 
 
 
 10 5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 VelocityX 
 
 
 VelocityX 
 
 
 
 
 
 VelocityY 
 
 
 VelocityY 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -3 
 
 
 
	<path stroke='black' d='M803.0,417.2 L807.5,417.2 M1442.6,417.2 L1438.1,417.2 M803.0,392.6 L807.5,392.6 M1442.6,392.6 L1438.1,392.6
 M803.0,375.2 L807.5,375.2 M1442.6,375.2 L1438.1,375.2 M803.0,361.7 L807.5,361.7 M1442.6,361.7 L1438.1,361.7
 M803.0,350.7 L807.5,350.7 M1442.6,350.7 L1438.1,350.7 M803.0,341.3 L807.5,341.3 M1442.6,341.3 L1438.1,341.3
 M803.0,333.3 L807.5,333.3 M1442.6,333.3 L1438.1,333.3 M803.0,326.1 L807.5,326.1 M1442.6,326.1 L1438.1,326.1
 M803.0,319.8 L812.0,319.8 M1442.6,319.8 L1433.6,319.8 '/> 
 10 -2 
 
 
 
	<path stroke='black' d='M803.0,277.8 L807.5,277.8 M1442.6,277.8 L1438.1,277.8 M803.0,253.3 L807.5,253.3 M1442.6,253.3 L1438.1,253.3
 M803.0,235.9 L807.5,235.9 M1442.6,235.9 L1438.1,235.9 M803.0,222.4 L807.5,222.4 M1442.6,222.4 L1438.1,222.4
 M803.0,211.3 L807.5,211.3 M1442.6,211.3 L1438.1,211.3 M803.0,202.0 L807.5,202.0 M1442.6,202.0 L1438.1,202.0
 M803.0,193.9 L807.5,193.9 M1442.6,193.9 L1438.1,193.9 M803.0,186.8 L807.5,186.8 M1442.6,186.8 L1438.1,186.8
 M803.0,180.4 L812.0,180.4 M1442.6,180.4 L1433.6,180.4 '/> 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 10 3 
 
 
 
 
 10 4 
 
 
 
 
 10 5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Pressure 
 
 
 Pressure

In [61]:
DoubleExtensions.LogLogRegressionSlope(numJ, L2norms.ExtractSubArrayShallow(0, -1).To1DArray())

-0.8002661208355437

In [62]:
DoubleExtensions.LogLogRegressionSlope(numJ, L2norms.ExtractSubArrayShallow(1, -1).To1DArray())

-0.8286619046506948

In [63]:
DoubleExtensions.LogLogRegressionSlope(numJ, L2norms.ExtractSubArrayShallow(2, -1).To1DArray())

-0.5207415637456786